# Planet combinations: five or more planets in one sign

Ported from the Colab `Planets_Combination.ipynb` (archived repository `astro-fusion/Astrology_Planetary_Data`). How often do 5, 6 or more of the nine grahas share a sign, and on which dates?

> **Sidereal correction.** The Colab originals read
> `Astrology_Planetary_Data` (now archived), whose CSVs hold only house
> occupancy per day for an unsourced ascendant and could not be reproduced
> with either a tropical or a Lahiri-sidereal calculation. This version reads
> the regenerated dataset (`libs/vedic_astrology_core/datasets`): sidereal
> Lahiri, mean node, Ketu = Rahu + 180, whole-sign. In whole-sign charts
> "same house" is the same as "same sign", so the sign is used and no
> ascendant is needed. Rahu and Ketu are always 180 degrees apart, so they
> never share a sign.

In [ ]:
# Parameters (papermill overrides these)
START_YEAR = 1600
END_YEAR = 2100
DATA_DIR = "../data/datasets/planetary"
MIN_PLANETS = 5   # planets in one sign
CLUSTER_DAYS = 3  # dates within this many days collapse to one event

## Load the dataset

One row per body per day at 12:00 UTC. Generate it with `python scripts/generate_planetary_dataset.py --years 1600-2100`.

In [ ]:
from pathlib import Path

import pandas as pd

from vedic_astrology_core.datasets.planetary import combined, generate

years = range(START_YEAR, END_YEAR + 1)
try:
    df = combined(years, data_dir=DATA_DIR)
except FileNotFoundError:
    # Regenerate offline with the verified engine (about 1 s per 3 years),
    # or download a century zip from the release page:
    # https://github.com/astro-fusion/astro_research-white-paper/releases
    generate(START_YEAR, END_YEAR, out_dir=DATA_DIR, fmt="parquet")
    df = combined(years, data_dir=DATA_DIR)
df["day"] = (df["jd"] + 0.5).astype(int)  # integer Julian day number
df["year"] = df["date"].str[:4].astype(int)
print(df.shape)
df.head()

## Sign occupancy per day

In [ ]:
signs = (
    df.groupby(["day", "year", "sign", "sign_name"])
    .agg(n=("body", "size"), bodies=("body", lambda s: " ".join(s)))
    .reset_index()
)
crowded = signs[signs["n"] >= MIN_PLANETS].copy()
print(f"{len(crowded)} (day, sign) rows with >= {MIN_PLANETS} planets")
crowded.head(10)

## How many (day, sign) combinations by size

In [ ]:
sizes = signs[signs['n'] >= 2].groupby('n').size().rename('days')
sizes.to_frame()

## Dates of the large combinations

The old notebook estimated dates from the row index; here every row carries its exact UTC date. Neighbouring dates are merged into one event.

In [ ]:
import numpy as np


def cluster_days(days, gap):
    """Group sorted integer days whose neighbours are at most ``gap`` apart.

    Returns one dict per cluster with its first, last and central day.
    """
    days = np.sort(np.unique(days))
    if len(days) == 0:
        return []
    breaks = np.where(np.diff(days) > gap)[0] + 1
    return [
        {"first": g[0], "last": g[-1], "centre": int(np.median(g)), "n": len(g)}
        for g in np.split(days, breaks)
    ]


def day_to_date(day):
    """ISO date of an integer Julian day number."""
    from vedic_astrology_core.utils.datetime_utils import julian_day_to_datetime

    return julian_day_to_datetime(day - 0.5).date().isoformat()

In [ ]:
events = cluster_days(crowded["day"].to_numpy(), CLUSTER_DAYS)
table = pd.DataFrame(
    [
        {
            "centre": day_to_date(e["centre"]),
            "first": day_to_date(e["first"]),
            "last": day_to_date(e["last"]),
            "days": e["n"],
        }
        for e in events
    ]
)
print(f"{len(table)} events in {START_YEAR}-{END_YEAR}")
table.head(20)